# Part 2 — E-Statement ETL

Source: `FSB Bank Transaction.xlsx`. Result: `f_estatement_trx` in PostgreSQL and `f_estatement_trx.xlsx` for comparison. Run the cells in order. The database load cell uses the connection details supplied for this project.

## 1. Data flow

![Data flow for the E-Statement datamart](part_two_data_flow.png)

Excel source (six sheets) → pandas extraction → fee filtering and master-data joins → e-statement table → PostgreSQL and Excel.

## 2. Extract data

Read every worksheet. Keep account, branch, and bank codes as text so joins and leading zeros work correctly.

In [1]:
import pandas as pd
import sqlalchemy as sa
from datetime import datetime, timedelta, timezone

# Extract data from all Excel sheets
def extract_data_from_xlsx(file_path):
    data = pd.read_excel(
        file_path,
        sheet_name = None,
        dtype = {'account_number': 'string', 'bank_code': 'string', 'branch_code': 'string'}
    )
    return(data)

In [2]:
# Data extraction process
data_extracted = extract_data_from_xlsx('FSB Bank Transaction.xlsx')
for key, value in data_extracted.items():
    print(f'{key}: {len(value):,} rows, {len(value.columns)} columns')

transaction: 1,640 rows, 10 columns
master_account: 123 rows, 7 columns
master_customer: 123 rows, 13 columns
master_transaction_code: 6 rows, 2 columns
master_bank_code: 38 rows, 3 columns
master_branch: 74 rows, 3 columns


## 3. Transform data

Exclude standalone monthly-fee rows. Pair ATM and transfer-fee rows to the main transaction by `transaction_reference`, then put the fee in `additional_amount`. Keep both debit and credit rows of internal transfers because they are separate account entries.

Join the four directly related master tables. Parse bank details from the remark. A transfer is `OVERBOOKING` when its counterparty account exists in `master_account`. Bank code `116` has two names in the source, so its name stays blank until that master-data issue is resolved.

In [3]:
# Data transformation process for converting transaction data into e-statement table
def transform_data(data):
    trx = data['transaction'].copy()
    account = data['master_account']
    customer = data['master_customer']
    trx_code = data['master_transaction_code']
    branch = data['master_branch']
    bank = data['master_bank_code']

    # Identify different types of fees in the transaction data
    is_monthly_fee = trx['transaction_code'].eq('TRX091')
    is_atm_fee = trx['transaction_code'].eq('TRX092')

    # Transfer fees are small debit rows paired with the main transfer
    group = trx.groupby('transaction_reference')
    is_transfer_fee = (
        trx['transaction_code'].eq('TRX002')
        & trx['debit_or_credit'].eq('D')
        & group['transaction_id'].transform('size').eq(2)
        & group['account_number'].transform('nunique').eq(1)
        & trx['amount'].isin([2500, 7500])
        & trx['amount'].lt(group['amount'].transform('max'))
    )
    fee = trx[is_atm_fee | is_transfer_fee]
    fee_by_reference = fee.groupby('transaction_reference')['amount'].sum()
    trx = trx[~(is_monthly_fee | is_atm_fee | is_transfer_fee)].copy()
    trx['additional_amount'] = trx['transaction_reference'].map(fee_by_reference).astype('Int64')

    # Merge master data, keep transactions without matching master records
    result = trx.merge(account, on='account_number', how='left')
    result = result.merge(customer, on='cif_number', how='left')
    result = result.merge(trx_code, on='transaction_code', how='left')
    result = result.merge(branch, on='branch_code', how='left')

    # Transaction date is in YYYYDDD format; time is in HHMMSS numeric format
    transaction_date = pd.to_datetime(result['transaction_date'].astype(str), format='%Y%j')
    transaction_time = result['transaction_time'].astype(str).str.zfill(6)
    result['transaction_datetime'] = pd.to_datetime(
        transaction_date.dt.strftime('%Y-%m-%d') + ' ' + transaction_time,
        format='%Y-%m-%d %H%M%S'
    )
    result['account_aging'] = (transaction_date - result['account_open_date']).dt.days.astype('Int64')

    # Transaction codes for e-statement
    remark = result['transaction_remark'].astype('string')
    remark_parts = remark.str.split('_', n=3, expand=True)
    remark_type = remark_parts[0]
    parsed_bank_code = remark_parts[1].where(remark_type.isin(['TRF', 'BIFAST', 'QRIS']))
    withdrawal_bank_code = remark.str.extract(r'^TARIK VIA\s+\S+\s+(\d{3})\s+', expand=False)
    result['acquirer_bank_code'] = parsed_bank_code.fillna(withdrawal_bank_code).replace('null', pd.NA)
    result['acquirer_account_number'] = remark_parts[2].where(remark_type.isin(['TRF', 'BIFAST']))
    result['acquirer_account_name'] = remark_parts[3].where(remark_type.isin(['TRF', 'BIFAST', 'QRIS']))

    result['transmode_code'] = 'OUTGOING'
    result.loc[result['debit_or_credit'].eq('C'), 'transmode_code'] = 'INCOMING'
    result.loc[result['transaction_code'].eq('TRX001'), 'transmode_code'] = 'SETOR'
    result.loc[result['transaction_code'].eq('TRX003'), 'transmode_code'] = 'TARIK'
    result.loc[result['transaction_code'].eq('TRX004') | remark_type.eq('QRIS'), 'transmode_code'] = 'PURCHASE'
    result.loc[remark_type.isin(['TRF', 'BIFAST']) & result['acquirer_account_number'].isin(account['account_number']), 'transmode_code'] = 'OVERBOOKING'

    # Bank code 116 has two names; do not choose one name randomly
    unique_bank = bank.drop_duplicates('bank_code', keep=False)[['bank_code', 'bank_name']]
    result = result.merge(unique_bank, left_on='acquirer_bank_code', right_on='bank_code', how='left')
    result = result.rename(columns={'bank_name': 'acquirer_bank_name'})

    result['credit_amount'] = result['amount'].where(result['debit_or_credit'].eq('C')).astype('Int64')
    result['debit_amount'] = result['amount'].where(result['debit_or_credit'].eq('D')).astype('Int64')
    current_time = datetime.now(timezone(timedelta(hours=7)))
    result['created_date'] = current_time.strftime('%d:%m:%y %H:%M:%S')

    columns = [
        'transaction_id', 'transaction_reference', 'transmode_code',
        'account_number', 'cif_number', 'customer_name', 'gender',
        'account_status', 'account_open_date', 'branch_code', 'branch_name',
        'account_aging', 'product_code', 'product_name', 'transaction_code',
        'transaction_desc', 'debit_or_credit', 'account_currency_type',
        'transaction_datetime', 'credit_amount', 'debit_amount',
        'acquirer_bank_code', 'acquirer_bank_name', 'acquirer_account_number',
        'acquirer_account_name', 'additional_amount', 'transaction_remark',
        'channel', 'created_date'
    ]
    return(result[columns])

In [4]:
# Data transformation process for converting transaction data into e-statement table
data_transformed = transform_data(data_extracted)
print(f'ETL Result: {len(data_transformed):,} rows, {len(data_transformed.columns)} columns')
display(data_transformed.head())

ETL Result: 1,202 rows, 29 columns


,transaction_id,transaction_reference,transmode_code,account_number,cif_number,customer_name,gender,account_status,account_open_date,branch_code,...,credit_amount,debit_amount,acquirer_bank_code,acquirer_bank_name,acquirer_account_number,acquirer_account_name,additional_amount,transaction_remark,channel,created_date
0,TRX21325483579612,100001020252132548357,SETOR,90000260801008087,CIF038315,Farah Simbolon,P,ACTIVE,2025-08-01,50007,...,5200000,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,SETORTUNAI VIA CASH DEPOSIT MACHINE 00047,ATM_CDM,30:09:26 06:00:47
1,TRX21335043336597,100001020252133504333,SETOR,90000260801002198,CIF092706,Kania Puspita,P,ACTIVE,2025-08-01,20001,...,4000000,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,SETORTUNAI VIA AGENT 00070,AGENT,30:09:26 06:00:47
2,TRX21343446066904,100001020252134344606,SETOR,90000260801006643,CIF062468,Dartono Rajasa,L,ACTIVE,2025-08-01,40008,...,7600000,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,SETORTUNAI VIA CASH RECYCLING MACHINE 00037,ATM_CRM,30:09:26 06:00:47
3,TRX21357196529986,100001020252135719652,SETOR,90000260801008644,CIF077350,Yunita Agustina,P,ACTIVE,2025-08-01,50005,...,3100000,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,SETORTUNAI VIA AGENT 00183,AGENT,30:09:26 06:00:47
4,TRX21439859637972,100001020252143985963,SETOR,90000260802006117,CIF043473,Wadi Putra,L,ACTIVE,2025-08-02,60004,...,8400000,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,SETORTUNAI VIA CASH DEPOSIT MACHINE 00070,ATM_CDM,30:09:26 06:00:47


## 4. Check the result

The output should contain only main transaction entries. The source has 1,640 rows: 137 monthly fees, 55 ATM fees, and 246 paired transfer fees are excluded, leaving 1,202 statement rows. The 301 paired fees remain visible as `additional_amount` on their main entries.

In [5]:
# Check the results of the data transformation
print('Transaction type counts:')
display(data_transformed['transmode_code'].value_counts().rename('rows').to_frame())
print('Additional charges applied:', data_transformed['additional_amount'].notna().sum())
print('Transactions without master account:', data_transformed['cif_number'].isna().sum())
print('Bank code without master name:', (
    data_transformed['acquirer_bank_code'].notna()
    & data_transformed['acquirer_bank_name'].isna()
).sum())
internal = data_extracted['transaction'][
    data_extracted['transaction']['transaction_remark'].str.startswith('TRF_456_')
]
print('Internal transfers with different debit and credit amounts:', (
    internal.groupby('transaction_reference')['amount'].nunique() > 1
).sum())
assert data_transformed['transaction_id'].is_unique
assert not data_transformed['transaction_code'].isin(['TRX091', 'TRX092']).any()
assert len(data_transformed) == 1202

Transaction type counts:


,rows
transmode_code,
PURCHASE,664
OUTGOING,246
SETOR,141
INCOMING,70
TARIK,61
OVERBOOKING,20


Additional charges applied: 301
Transactions without master account: 14
Bank code without master name: 80
Internal transfers with different debit and credit amounts: 4


## 5. Export for comparison

Save the expected 29 columns to an Excel file in this project folder. Account and bank identifiers remain text.

In [6]:
# Process of saving the results to Excel
data_transformed.to_excel('f_estatement_trx.xlsx', index=False)
print('File f_estatement_trx.xlsx has been successfully created')

File f_estatement_trx.xlsx has been successfully created


## 6. Load to PostgreSQL

The database details are `localhost:5432`, database `db_mini_project_3_elt_project`, user `postgres`, and an empty password. Run the next cell when that database is available. It replaces the existing `f_estatement_trx` table, matching the loading pattern in the reference notebooks.

The bank lookup is incomplete for some codes, 14 output rows have no matching account master, and four internal-transfer pairs have unequal debit and credit amounts. Review these source exceptions before treating the datamart as production data.

In [7]:
# Process of getting connection to the database
def connect_to_database():
    username = 'postgres'
    password = ''
    host = 'localhost'
    port = '5432'
    database = 'db_mini_project_3_elt_project'

    engine = sa.create_engine(
        f'postgresql+psycopg://{username}:{password}@{host}:{port}/{database}'
    )
    return(engine)

# Process of loading data / saving data to the database
def load_data(data, table_name, engine):
    with engine.begin() as connection:
        data.to_sql(
            table_name,
            connection,
            if_exists = 'replace',
            index = False
        )

In [8]:
# Run this when PostgreSQL is already active
engine = connect_to_database()
load_data(
    data = data_transformed,
    table_name = 'f_estatement_trx',
    engine = engine
)
print('Table f_estatement_trx has been successfully saved to PostgreSQL')

Table f_estatement_trx has been successfully saved to PostgreSQL


## 7. Verify the PostgreSQL table

Read back five rows and count the saved records. This output can be captured together with the ETL code for the classroom submission.


In [9]:
# Check the resulting table in PostgreSQL
with engine.connect() as connection:
    row_count = pd.read_sql('SELECT COUNT(*) AS total_rows FROM f_estatement_trx', connection)
    data_saved = pd.read_sql('SELECT * FROM f_estatement_trx LIMIT 5', connection)
display(row_count)
display(data_saved)


,total_rows
0,1202


,transaction_id,transaction_reference,transmode_code,account_number,cif_number,customer_name,gender,account_status,account_open_date,branch_code,...,credit_amount,debit_amount,acquirer_bank_code,acquirer_bank_name,acquirer_account_number,acquirer_account_name,additional_amount,transaction_remark,channel,created_date
0,TRX21325483579612,100001020252132548357,SETOR,90000260801008087,CIF038315,Farah Simbolon,P,ACTIVE,2025-08-01,50007,...,5200000,None,None,None,None,None,None,SETORTUNAI VIA CASH DEPOSIT MACHINE 00047,ATM_CDM,30:09:26 06:00:47
1,TRX21335043336597,100001020252133504333,SETOR,90000260801002198,CIF092706,Kania Puspita,P,ACTIVE,2025-08-01,20001,...,4000000,None,None,None,None,None,None,SETORTUNAI VIA AGENT 00070,AGENT,30:09:26 06:00:47
2,TRX21343446066904,100001020252134344606,SETOR,90000260801006643,CIF062468,Dartono Rajasa,L,ACTIVE,2025-08-01,40008,...,7600000,None,None,None,None,None,None,SETORTUNAI VIA CASH RECYCLING MACHINE 00037,ATM_CRM,30:09:26 06:00:47
3,TRX21357196529986,100001020252135719652,SETOR,90000260801008644,CIF077350,Yunita Agustina,P,ACTIVE,2025-08-01,50005,...,3100000,None,None,None,None,None,None,SETORTUNAI VIA AGENT 00183,AGENT,30:09:26 06:00:47
4,TRX21439859637972,100001020252143985963,SETOR,90000260802006117,CIF043473,Wadi Putra,L,ACTIVE,2025-08-02,60004,...,8400000,None,None,None,None,None,None,SETORTUNAI VIA CASH DEPOSIT MACHINE 00070,ATM_CDM,30:09:26 06:00:47
